In [1]:
import json
import pandas as pd
import numpy as np

occlusion_map = {
      'NOT_OCCLUDED': 0.29,
      'PARTIALLY_OCCLUDED': 0.64,
      'MOSTLY_OCCLUDED': 0.98,
      '': 0.2  # default for blank or missing
}

def preprocess_single_file(file_path):
    # Load JSON data
    with open(file_path, 'r') as f:
        data = json.load(f)

    # Initialize data containers
    transforms_data = []
    objects_data = []

    def getPDRandThroughput(tm):
        transformation_matrix = np.array(tm)
        translation = transformation_matrix[:3, 3]

        distance_m = np.linalg.norm(translation)

        def estimate_pdr(distance, max_distance=500):
            # Assume PDR ~1 at 0m, drops to ~0 at max_distance
            return np.clip(np.exp(-distance / (max_distance / 3)), 0, 1)

        def estimate_throughput(pdr, max_rate_mbps=15):
            return pdr * max_rate_mbps

        pdr = estimate_pdr(distance_m)
        throughput = estimate_throughput(pdr)
        return pdr, throughput
        

    # Extract transforms
    for frame in data['openlabel']['frames'].values():
        transforms = frame['frame_properties']['transforms']
        for tf_name, tf_info in transforms.items():
            transformationMatrix = np.array(tf_info['transform_src_to_dst']['matrix4x4'])
            pdr, throughput = getPDRandThroughput(transformationMatrix)
            transforms_data.append({
                'source_sensor': tf_info['src'],
                'target_sensor': tf_info['dst'],
                'transformation_matrix': transformationMatrix,
                'pdr':pdr,
                'throughput':throughput
            })

    # Extract objects
    for frame_id, frame_data in data['openlabel']['frames'].items():
        for obj_id, obj_info in frame_data['objects'].items():
            cuboid = obj_info['object_data']['cuboid']['val']
            attributes = obj_info['object_data']['cuboid']['attributes']

            # Parse attributes
            attr_dict = {'sensor_id': None, 'occlusion_level': None, 'num_points': -1, 'score': -1}
            for text_attr in attributes['text']:
                if text_attr['name'] == 'sensor_id':
                    attr_dict['sensor_id'] = text_attr['val']
                if text_attr['name'] == 'occlusion_level':
                    attr_dict['occlusion_level'] = occlusion_map.get(text_attr['val'],0.2)
            for num_attr in attributes['num']:
                if num_attr['name'] == 'num_points':
                    attr_dict['num_points'] = num_attr['val']
                if num_attr['name'] == 'score':
                    occlusion_factor = attr_dict['occlusion_level']
                    density_score_calculation = (attr_dict['num_points'] / (cuboid[7] * cuboid[8] * cuboid[9])) * (1- occlusion_factor )
                    attr_dict['score'] = num_attr['val'] if num_attr['val'] > 0 else density_score_calculation

            # Parse cuboid values (assuming format: [x, y, z, qx, qy, qz, qw, length, width, height])
            objects_data.append({
                'frame_id': frame_id,
                'object_id': obj_id,
                'object_type': obj_info['object_data']['type'],
                'x': cuboid[0],
                'y': cuboid[1],
                'z': cuboid[2],
                'qx': cuboid[3],
                'qy': cuboid[4],
                'qz': cuboid[5],
                'qw': cuboid[6],
                'length': cuboid[7],
                'width': cuboid[8],
                'height': cuboid[9],
                **attr_dict
            })

    

    # Create DataFrames
    transforms_df = pd.DataFrame(transforms_data)
    objects_df = pd.DataFrame(objects_data)

    # Add timestamp
    timestamp = next(iter(data['openlabel']['frames'].values()))['frame_properties']['timestamp']
    objects_df['timestamp'] = timestamp

    return transforms_df, objects_df


def process_multiple_files(file_list):
    all_transforms = []
    all_objects = []
    
    for file_path in file_list:
        t_df, o_df = preprocess_single_file(file_path)
        all_transforms.append(t_df)
        all_objects.append(o_df)
    
    return pd.concat(all_transforms), pd.concat(all_objects)



file_list = [f"LIDARJSONS/preprocessLidar_{i:02}.json" for i in range(1,28)]
transforms_full, objects_full = process_multiple_files(file_list)

## Preprocess For DENM Messages using Channel Busy Ratio(CBR) throughput

In [2]:
objects_full = pd.merge(transforms_full, objects_full,left_on='target_sensor', right_on='sensor_id', how='inner')
objects_full = objects_full.drop(columns=['source_sensor','transformation_matrix','target_sensor'])
objects_full['throughput'] =  (objects_full['throughput']/4.8)* (1 - objects_full['occlusion_level'])

In [3]:
frame_stats = objects_full.groupby(['frame_id', 'timestamp']).agg(
    num_messages=('object_id', 'count'),
    avg_throughput=('throughput', 'mean')
).reset_index()

frame_stats = frame_stats.sort_values(['frame_id', 'timestamp'])
frame_stats['prev_ts'] = frame_stats.groupby('frame_id')['timestamp'].shift(1)

frame_stats['measurement_interval'] = np.where(
    frame_stats['prev_ts'].notnull(),
    frame_stats['timestamp'] - frame_stats['prev_ts'],
    0.1  # Default 100ms when using first ....[4]
)


frame_stats['cbr_objects'] = np.minimum(
    (frame_stats['num_messages'] * 800 * 8) / (6e6 * frame_stats['measurement_interval']),
    1.0  # Cap at 1.0 (100%) ....[4]
)


frame_stats['cbr_throughput'] = (frame_stats['avg_throughput'] * 8) / (6e6 * frame_stats['measurement_interval'])

denm_conditions = [
    frame_stats['cbr_objects'] < 0.35,
    frame_stats['cbr_objects'] <= 25
]
denm_choices = [10,5]

frame_stats['denm_frequency'] =np.select(denm_conditions,denm_choices,default=2)
merged_objectdf = pd.merge(objects_full, frame_stats[['frame_id','cbr_objects','denm_frequency']], on='frame_id',suffixes=('', '_new'))
merged_objectdf.head(2)
objects_full['cbr_objects'] = merged_objectdf['cbr_objects']
objects_full['denm_frequency'] = merged_objectdf['denm_frequency']

In [4]:
newData= objects_full.sort_values("timestamp")
n = len(objects_full)
train = objects_full.iloc[:int(0.6*n)]
val = objects_full.iloc[int(0.6*n):int(0.8*n)]
test = objects_full.iloc[int(0.8*n):]

In [5]:
train.to_csv("train_new.csv", index=False)
print(f"Train of size: {len(train)} is added")
val.to_csv("val_new.csv", index=False)
print(f"Validation of size: {len(val)} is added")
test.to_csv("test_new.csv", index=False)
print(f"Test of size: {len(test)} is added")

Train of size: 4665 is added
Validation of size: 1555 is added
Test of size: 1556 is added
